# 4-model 완전 Nested Stacking

Notebook 2의 저장된 OOF 기반 meta-selection을 대체합니다. 후보는 `EXP-639`, `EXP-545`, `EXP-127`, `EXP-334` 네 개로 사전 고정하며, 과거 OOF 파일을 입력으로 읽지 않습니다.

각 outer fold 안에서 네 base model을 inner 4-fold로 다시 학습해 inner-OOF를 만들고 ExtraTrees를 학습합니다. 이후 base model을 outer-train 전체로 다시 fit한 뒤, 한 번도 보지 않은 outer-validation을 예측합니다. TF-IDF vocabulary/IDF, class cosine profile, Hotspot-12도 각 fit index 범위에서만 학습됩니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

root_path = Path.cwd().resolve()
if root_path.name == 'notebooks':
    root_path = root_path.parent
if not (root_path / 'PROJECT_CONTEXT.md').exists():
    raise FileNotFoundError('저장소 루트 또는 notebooks 디렉터리에서 실행하세요.')
sys.path.insert(0, str(root_path / 'src'))
sys.path.insert(0, str(root_path / 'scripts'))
print('root:', root_path)

## 1. 후보 모델 고정

`main_features`는 사람이 붙인 선택 점수가 아니라 계보를 설명하기 위한 메모입니다. 실제 앙상블 판단 근거는 아래에서 새로 생성되는 inner/outer OOF, 오류 불일치율과 fit-scope 감사표입니다.

In [ ]:
model_info = [
    {
        'experiment': 'EXP-639',
        'file_name': 'exp639_parser_v4_hotspot12',
        'algorithm': 'XGBoost',
        'main_features': 'parser-v4 class cosine + fold-train Hotspot-12',
    },
    {
        'experiment': 'EXP-545',
        'file_name': 'exp545_hierarchical_tfidf_linear',
        'algorithm': 'LinearSVC',
        'main_features': 'canonical mutation hierarchical TF-IDF',
    },
    {
        'experiment': 'EXP-127',
        'file_name': 'exp127_catboost_v1',
        'algorithm': 'CatBoost',
        'main_features': 'frozen Feature Spec v1 sparse matrix',
    },
    {
        'experiment': 'EXP-334',
        'file_name': 'exp334_exp285_isoform_residue_mask',
        'algorithm': 'XGBoost',
        'main_features': 'isoform residue mask + pathway mutation types',
    },
]
model_table = pd.DataFrame(model_info)
display(model_table)

## 2. Outer/inner 격리 계약 확인

canonical outer 5-fold에서 outer-train을 구성하는 나머지 네 canonical fold를 그대로 inner fold로 사용합니다. 따라서 각 outer-train 행은 inner-validation에 정확히 한 번 나타납니다.

In [ ]:
from open_cancer.fully_nested_stacking import build_outer_inner_splits, index_sha256

fold_frame = pd.read_csv(root_path / 'data/splits/stratified_5fold_seed42.csv')
fold_values = fold_frame['fold'].to_numpy(dtype=np.int32)
split_rows = []
for split in build_outer_inner_splits(fold_values):
    split_rows.append({
        'outer_fold': split.outer_fold,
        'inner_fold': split.inner_fold,
        'fit_rows': len(split.fit_indices),
        'inner_validation_rows': len(split.validation_indices),
        'outer_validation_rows': len(split.outer_validation_indices),
        'fit_outer_valid_overlap': len(np.intersect1d(split.fit_indices, split.outer_validation_indices)),
        'fit_inner_valid_overlap': len(np.intersect1d(split.fit_indices, split.validation_indices)),
        'fit_index_sha256': index_sha256(split.fit_indices),
    })
split_audit = pd.DataFrame(split_rows)
assert len(split_audit) == 20
assert split_audit[['fit_outer_valid_overlap', 'fit_inner_valid_overlap']].to_numpy().sum() == 0
display(split_audit)

## 3. 구현해야 할 핵심 루프 실행

- `audit`: split만 즉시 검사
- `smoke`: 네 실제 adapter를 트리 2개로 한 번씩 검사
- `full`: 80 inner fit + 20 outer refit + 4 final refit 실행

`auto`는 CUDA가 있으면 GPU, 없으면 CPU를 사용합니다. EXP-127 full CPU 실행은 매우 오래 걸리므로 CUDA 머신이 권장됩니다. 완료된 fit은 index/config hash 기반 캐시에 저장되어 중단 후 같은 셀을 다시 실행하면 이어집니다.

In [ ]:
EXECUTION_MODE = 'full'  # 'audit', 'smoke', 'full'
DEVICE_POLICY = 'gpu'    # RunPod RTX 5090 full run

command = [
    sys.executable,
    str(root_path / 'scripts/run_baseline3_four_model_fully_nested.py'),
    '--mode', EXECUTION_MODE,
    '--device-policy', DEVICE_POLICY,
]
print(' '.join(command), flush=True)
subprocess.run(command, cwd=root_path, check=True)

## 4. 완전 nested 결과와 fit-scope 감사

이 절은 `full` 실행 후 생성된 결과만 읽습니다. `fit_protected_overlap`과 `fit_predict_overlap`은 모두 0이어야 합니다.

In [ ]:
result_dir = root_path / 'reports/analysis/baseline3_four_model_fully_nested'
result_path = result_dir / 'result.json'
audit_path = result_dir / 'fit_scope_audit.csv'
if not result_path.exists():
    print('full 결과가 아직 없습니다. EXECUTION_MODE를 full로 실행하세요.')
else:
    result = json.loads(result_path.read_text(encoding='utf-8'))
    audit = pd.read_csv(audit_path)
    assert result['historical_oof_loaded'] is False
    assert result['base_fit_count'] == result['expected_base_fit_count'] == 104
    assert (audit['fit_protected_overlap'] == 0).all()
    assert (audit.loc[audit['fit_predict_overlap'].notna(), 'fit_predict_overlap'] == 0).all()
    display(pd.DataFrame(result['folds']))
    display(pd.Series(result['oof'], name='value').to_frame())
    display(audit.groupby(['stage', 'model']).agg(fits=('model', 'size'), cache_hits=('cache_hit', 'sum'), max_protected_overlap=('fit_protected_overlap', 'max')))

## 5. 실제 OOF 기반 다양성 모니터

색 범위를 데이터의 최솟값·최댓값에 억지로 맞추지 않습니다. 라벨 불일치율과 동시 오류율은 이론적 범위 `[0, 1]`, 확률 상관은 `[-1, 1]`로 고정합니다. 이 차트는 후보 선택에 다시 사용하지 않는 사후 진단입니다.

In [ ]:
if result_path.exists():
    true_labels = pd.read_csv(root_path / 'data/raw/train.csv', usecols=['SUBCLASS'])['SUBCLASS'].to_numpy()
    prediction_labels = {}
    probability_vectors = {}
    for experiment in model_table['experiment']:
        suffix = experiment.lower().replace('-', '')
        frame = pd.read_csv(root_path / f'oof/baseline3_four_model_fully_nested_{suffix}.csv')
        probability = frame[[column for column in frame if column.startswith('PROBA_')]].to_numpy()
        probability_vectors[experiment] = probability.ravel()
        prediction_labels[experiment] = np.asarray([column.removeprefix('PROBA_') for column in frame if column.startswith('PROBA_')])[probability.argmax(axis=1)]

    experiments = model_table['experiment'].tolist()
    disagreement = pd.DataFrame(index=experiments, columns=experiments, dtype=float)
    double_fault = disagreement.copy()
    probability_correlation = disagreement.copy()
    for left in experiments:
        for right in experiments:
            disagreement.loc[left, right] = np.mean(prediction_labels[left] != prediction_labels[right])
            double_fault.loc[left, right] = np.mean((prediction_labels[left] != true_labels) & (prediction_labels[right] != true_labels))
            probability_correlation.loc[left, right] = np.corrcoef(probability_vectors[left], probability_vectors[right])[0, 1]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    sns.heatmap(disagreement, annot=True, fmt='.3f', vmin=0, vmax=1, cmap='Blues', ax=axes[0])
    axes[0].set_title('Prediction disagreement')
    sns.heatmap(double_fault, annot=True, fmt='.3f', vmin=0, vmax=1, cmap='Reds', ax=axes[1])
    axes[1].set_title('Double fault')
    sns.heatmap(probability_correlation, annot=True, fmt='.3f', vmin=-1, vmax=1, center=0, cmap='vlag', ax=axes[2])
    axes[2].set_title('Probability correlation')
    plt.tight_layout()
    plt.show()

## 6. 제출 파일

최종 파일은 `submissions/baseline3_four_model_fully_nested.csv`입니다. 전체 train의 base OOF로 final ExtraTrees를 학습하고, 네 base model을 전체 train으로 다시 학습한 test 확률만 사용합니다.

In [ ]:
submission_path = root_path / 'submissions/baseline3_four_model_fully_nested.csv'
if submission_path.exists():
    submission = pd.read_csv(submission_path)
    sample = pd.read_csv(root_path / 'data/raw/sample_submission.csv')
    assert submission.columns.tolist() == ['ID', 'SUBCLASS']
    assert submission['ID'].equals(sample['ID'])
    assert set(submission['SUBCLASS']) <= set([
        'ACC','BLCA','BRCA','CESC','COAD','DLBC','GBMLGG','HNSC','KIPAN','KIRC','LAML','LGG','LIHC',
        'LUAD','LUSC','OV','PAAD','PCPG','PRAD','SARC','SKCM','STES','TGCT','THCA','THYM','UCEC'
    ])
    print(submission_path)
    display(submission.head())
else:
    print('full 실행 후 제출 파일이 생성됩니다.')